# Week 11 — 时序 Transformer 变体（MVP v0.5）：PatchTST

**目标**：在 L=256 长序列合成交易数据上，实现 PatchTST backbone，并与 vanilla Transformer、Informer 风格的 ProbSparse attention 做「参数量 / 显存 / 训练时长 / AUC-PR」四维对比。

**产出**：
- 三方 backbone 对比表（参数量 / peak GPU mem / 每 epoch 壁钟 / AUC-PR）
- 三方训练曲线
- 反思笔记：PatchTST 为什么有效？什么时候该选它？

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

## 1. 背景：vanilla attention 为什么在长序列上吃紧？

Scaled dot-product attention 的单层复杂度是 `O(L²·d)`（`L` 是序列长度，`d` 是 hidden dim）。当 `L=256` 时，`L²=65536`，QK^T 矩阵单 head、单 sample、FP32 已占 256 KB；再叠 batch / head / layer，显存几乎按 `L²` 线性消耗。

业界两条主流路线：

| 维度 | Vanilla | Informer (AAAI 2021 Best) | PatchTST (ICLR 2023) |
|------|---------|---------------------------|----------------------|
| Attention 计算对象 | 所有 token 对 | **Top-u 稀疏 query**（ProbSparse，按 `M(q_i, K)` 筛选） | **Patch**（长度 `P=16` 的窗口聚合成一个 token） |
| 复杂度 | `O(L²·d)` | `O(L · ln L · d)` | `O((L/S)²·d)`（`S` 是 stride） |
| 通道处理 | 多变量共享 attention | 多变量共享 attention | **Channel-independence**：每个变量独立跑 encoder |
| Decoder | 逐步自回归 | **生成式一次出全长度** | 线性头直接出预测 |
| 额外 trick | — | Self-attention distilling（Conv+MaxPool 降采样） | Instance norm（RevIN 思路） |

**本周要亲手做三件事**：
1. 写 PatchTST backbone（约 120 行，核心是 patching + channel-independence）。
2. 写 Informer 风格的 ProbSparse attention mini-ablation（约 50 行），只保留 top-u query 选择，用来感受稀疏化效果。
3. 在同一份 L=256 数据上训三个模型，产出对比表。

> 参考：Zhou et al., *Informer*, AAAI 2021；Nie et al., *A Time Series is Worth 64 Words (PatchTST)*, ICLR 2023。

## 2. 数据：长序列合成交易（L=256，随机位置注入异常）

沿用前几周的合成交易数据生成器，但把每个用户的序列长度拉到 256（≈ 前几周 L=64 的 4 倍），并在每条异常序列的随机位置注入一小段反常 pattern，以考验模型能否在长上下文里定位异常。

- 特征数 `F=8`（金额、时间差、地域编码、设备编码、金额 z-score、是否夜间、金额与历史均值比、通道编码）
- 正常序列：AR(1) 生成，加弱噪声
- 异常序列：先 AR(1)，再在 `[t0, t0+span]` 区间加入 spike + 通道偏置；`span ∈ [8, 24]`
- 正负比 = 95 : 5（在长序列上欺诈更稀疏）

In [ ]:
import math, time, copy
import numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

# ---- data config ----
N_USERS    = 2000      # total sequences
SEQ_LEN    = 256       # L
N_FEATS    = 8         # F
FRAUD_RATE = 0.05
rng = np.random.default_rng(SEED)

def _ar1(n, phi=0.85, sigma=1.0):
    x = np.zeros(n, dtype=np.float32)
    for t in range(1, n):
        x[t] = phi * x[t-1] + rng.normal(0, sigma)
    return x

def make_sequence(is_fraud):
    x = np.stack([_ar1(SEQ_LEN, phi=rng.uniform(0.6, 0.95)) for _ in range(N_FEATS)], axis=1)
    x += rng.normal(0, 0.2, x.shape).astype(np.float32)
    if is_fraud:
        span = rng.integers(8, 24)
        t0   = rng.integers(0, SEQ_LEN - span)
        ch   = rng.integers(0, N_FEATS)
        x[t0:t0+span, ch] += rng.uniform(3.0, 5.0)            # spike in one channel
        x[t0:t0+span, :] += rng.normal(0.5, 0.3, (span, N_FEATS)).astype(np.float32)
    return x

labels = (rng.random(N_USERS) < FRAUD_RATE).astype(np.int64)
X = np.stack([make_sequence(bool(y)) for y in labels], axis=0)   # (N, L, F)
print('X:', X.shape, ' fraud count:', labels.sum(), '/', len(labels))

In [ ]:
# ---- train/val split + standardize per feature on train ----
idx = rng.permutation(N_USERS)
split = int(N_USERS * 0.8)
tr_idx, va_idx = idx[:split], idx[split:]
Xtr, Xva = X[tr_idx], X[va_idx]
ytr, yva = labels[tr_idx], labels[va_idx]

mu  = Xtr.reshape(-1, N_FEATS).mean(0, keepdims=True)
std = Xtr.reshape(-1, N_FEATS).std(0, keepdims=True) + 1e-6
Xtr = ((Xtr - mu) / std).astype(np.float32)
Xva = ((Xva - mu) / std).astype(np.float32)

to_ds = lambda a, b: TensorDataset(torch.from_numpy(a), torch.from_numpy(b))
train_loader = DataLoader(to_ds(Xtr, ytr), batch_size=32, shuffle=True,  drop_last=True)
val_loader   = DataLoader(to_ds(Xva, yva), batch_size=64, shuffle=False)

print(f'train seqs={len(Xtr)}  val seqs={len(Xva)}  fraud in val={yva.sum()}')

## 3. PatchTST backbone（~120 行）

核心两步：

1. **Channel-independence**：输入 `(B, L, F)` reshape 成 `(B*F, L, 1)`，每个变量独立过 Transformer，encoder 不再做跨变量混合，缓解「特征量纲 / 频域差异互相污染」的问题。
2. **Patching**：把长度 `L` 的序列切成长度 `P=16`、stride `S=8` 的 patch，得到 `N = (L-P)/S + 1 = 31` 个 patch。每个 patch 线性映射到 `d_model`，再跑标准 Transformer Encoder。Attention 计算量从 `O(L²)` 降到 `O(N²)`，N=31 对比 L=256 接近 8x 缩减。

> 参考实现：`https://github.com/yuqinie98/PatchTST`；论文 § 3.1-3.2。

In [ ]:
# ---- PatchTST ----
class PatchEmbed(nn.Module):
    def __init__(self, patch_len, stride, d_model):
        super().__init__()
        self.patch_len = patch_len
        self.stride    = stride
        self.proj      = nn.Linear(patch_len, d_model)

    def forward(self, x):                       # x: (B*F, L, 1)
        x = x.squeeze(-1)                       # (B*F, L)
        # unfold into patches: (B*F, n_patches, patch_len)
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        return self.proj(x)                     # (B*F, N, d_model)


class PatchTST(nn.Module):
    """Channel-independent patching Transformer for binary classification."""
    def __init__(self, n_feats, seq_len, patch_len=16, stride=8,
                 d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.n_feats   = n_feats
        self.patch_len = patch_len
        self.stride    = stride
        n_patches      = (seq_len - patch_len) // stride + 1

        self.embed   = PatchEmbed(patch_len, stride, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # per-channel head → global pool over channels → classifier
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):                        # x: (B, L, F)
        B, L, F_ = x.shape
        # Channel-independence: fold F into batch
        x = x.permute(0, 2, 1).reshape(B * F_, L, 1)     # (B*F, L, 1)
        x = self.embed(x) + self.pos_emb                  # (B*F, N, d)
        x = self.encoder(x)                               # (B*F, N, d)
        x = x.mean(dim=1)                                 # (B*F, d) mean over patches
        logit_per_channel = self.head(x).squeeze(-1)      # (B*F,)
        logit_per_channel = logit_per_channel.view(B, F_) # (B, F)
        return logit_per_channel.mean(dim=1)              # (B,) pool over channels


m = PatchTST(N_FEATS, SEQ_LEN).to(device)
print(m)
print('params:', sum(p.numel() for p in m.parameters()))

## 4. ProbSparse attention mini-ablation（~50 行）

Informer 的核心洞察：大多数 attention query 的分布接近 uniform，不带多少信息。用 KL 距离衡量 query `q_i` 相对 uniform 的偏离度：

`M(q_i, K) ≈ max_j (q_i · k_j^T / √d) − mean_j (q_i · k_j^T / √d)`

选 top-`u = c · ln(L)` 个「活跃」 query 真算 softmax，其它 query 直接用 V 的均值填回。这里写一个最小可用版本，仅 attention 层替换 PyTorch 的 MHA；encoder block 其它部分沿用 `nn.TransformerEncoderLayer` 模板。

> 参考：Zhou et al., *Informer*, AAAI 2021，§ 3.2 "ProbSparse Self-attention"。

In [ ]:
# ---- ProbSparse attention (minimal ablation) ----
class ProbSparseAttention(nn.Module):
    def __init__(self, d_model, n_heads, factor=5, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.factor  = factor                       # c in paper
        self.qkv     = nn.Linear(d_model, d_model * 3)
        self.out     = nn.Linear(d_model, d_model)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):                           # x: (B, L, d)
        B, L, _ = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.d_head)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)        # (B, h, L, d_h)

        # Sparsity measure: for each q_i, max(q_i K^T) - mean(q_i K^T)
        scores_full = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        M = scores_full.max(dim=-1).values - scores_full.mean(dim=-1)   # (B, h, L)

        u = max(1, int(self.factor * math.log(L)))
        top_idx = M.topk(u, dim=-1).indices                              # (B, h, u)

        # gather top-u queries and compute full softmax only for them
        gather_idx = top_idx.unsqueeze(-1).expand(-1, -1, -1, self.d_head)
        q_top = q.gather(2, gather_idx)                                  # (B, h, u, d_h)
        attn  = torch.softmax(
            torch.matmul(q_top, k.transpose(-2, -1)) / math.sqrt(self.d_head),
            dim=-1)
        out_top = torch.matmul(self.drop(attn), v)                       # (B, h, u, d_h)

        # Non-selected queries → mean(V) (Informer trick)
        mean_v = v.mean(dim=2, keepdim=True).expand(-1, -1, L, -1).clone()
        mean_v.scatter_(2, gather_idx, out_top)
        out = mean_v.transpose(1, 2).reshape(B, L, self.d_model)
        return self.out(out)


class ProbSparseEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = ProbSparseAttention(d_model, n_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Linear(d_model * 4, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.drop(self.attn(x)))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class InformerLite(nn.Module):
    """Token = time step (multivariate fused), attention replaced with ProbSparse."""
    def __init__(self, n_feats, seq_len, d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_feats, d_model)
        self.pos_emb   = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        self.blocks = nn.ModuleList([
            ProbSparseEncoderBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):                               # x: (B, L, F)
        x = self.input_proj(x) + self.pos_emb
        for blk in self.blocks:
            x = blk(x)
        return self.head(x.mean(dim=1)).squeeze(-1)     # (B,)


m = InformerLite(N_FEATS, SEQ_LEN).to(device)
print('InformerLite params:', sum(p.numel() for p in m.parameters()))

## 5. Vanilla Transformer baseline（沿用 W07/W08 模式）

Token = 时间步（多变量共享 attention），标准 `nn.TransformerEncoder`。这是 `O(L²·d)` 的参照组。

In [ ]:
class VanillaTransformer(nn.Module):
    def __init__(self, n_feats, seq_len, d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_feats, d_model)
        self.pos_emb    = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        x = self.input_proj(x) + self.pos_emb
        x = self.encoder(x)
        return self.head(x.mean(dim=1)).squeeze(-1)


m = VanillaTransformer(N_FEATS, SEQ_LEN).to(device)
print('VanillaTransformer params:', sum(p.numel() for p in m.parameters()))

## 6. 统一训练 + 基准测量

同一 `train_one()` 跑三个模型：
- 同 epoch 数、同 lr、同 batch size、同种子
- 用 `torch.cuda.max_memory_allocated()` 统计 peak GPU mem
- 用壁钟时间统计每 epoch 时长
- 验证集 AUC-PR 作为效果指标

In [ ]:
def train_one(model, epochs=6, lr=1e-3, log_prefix=''):
    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    pos_weight = torch.tensor([(1 - FRAUD_RATE) / FRAUD_RATE], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    history = {'train_loss': [], 'val_aucpr': [], 'epoch_sec': []}
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    for ep in range(epochs):
        t0 = time.time()
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.float().to(device)
            opt.zero_grad()
            logit = model(xb)
            loss  = loss_fn(logit, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        epoch_sec = time.time() - t0

        model.eval()
        scores, ys = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                s  = torch.sigmoid(model(xb)).cpu().numpy()
                scores.append(s); ys.append(yb.numpy())
        aucpr = average_precision_score(np.concatenate(ys), np.concatenate(scores))

        history['train_loss'].append(float(np.mean(losses)))
        history['val_aucpr'].append(float(aucpr))
        history['epoch_sec'].append(epoch_sec)
        print(f'[{log_prefix}] ep{ep+1}/{epochs}  loss={np.mean(losses):.4f}  '
              f'val AUC-PR={aucpr:.4f}  time={epoch_sec:.1f}s')

    peak_mem_mb = (torch.cuda.max_memory_allocated() / 1024**2) if device.type == 'cuda' else 0.0
    param_cnt   = sum(p.numel() for p in model.parameters())
    return history, peak_mem_mb, param_cnt

In [ ]:
EPOCHS = 6

print('=== Vanilla Transformer ===')
m_van = VanillaTransformer(N_FEATS, SEQ_LEN)
hist_van, mem_van, n_van = train_one(m_van, epochs=EPOCHS, log_prefix='vanilla')

print('\n=== Informer-lite (ProbSparse) ===')
if device.type == 'cuda': torch.cuda.empty_cache()
m_prob = InformerLite(N_FEATS, SEQ_LEN)
hist_prob, mem_prob, n_prob = train_one(m_prob, epochs=EPOCHS, log_prefix='probsparse')

print('\n=== PatchTST ===')
if device.type == 'cuda': torch.cuda.empty_cache()
m_pst  = PatchTST(N_FEATS, SEQ_LEN, patch_len=16, stride=8)
hist_pst, mem_pst, n_pst = train_one(m_pst, epochs=EPOCHS, log_prefix='patchtst')

## 7. 对比表（请把打印出的数字填入本周 README）

In [ ]:
import pandas as pd

def summarize(name, hist, mem, n_params):
    return {
        'backbone'         : name,
        'params'           : n_params,
        'peak_mem_MB'      : round(mem, 1),
        'epoch_sec_mean'   : round(float(np.mean(hist['epoch_sec'])), 2),
        'final_val_AUCPR'  : round(hist['val_aucpr'][-1], 4),
        'best_val_AUCPR'   : round(max(hist['val_aucpr']), 4),
    }

tbl = pd.DataFrame([
    summarize('VanillaTransformer', hist_van,  mem_van,  n_van),
    summarize('InformerLite',       hist_prob, mem_prob, n_prob),
    summarize('PatchTST',           hist_pst,  mem_pst,  n_pst),
])
tbl

## 8. 训练曲线（三方叠画）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in [('vanilla', hist_van), ('probsparse', hist_prob), ('patchtst', hist_pst)]:
    axes[0].plot(hist['train_loss'], label=name, marker='o')
    axes[1].plot(hist['val_aucpr'],  label=name, marker='o')
axes[0].set_title('Train loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].set_title('Val AUC-PR'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()

## 9. 保存 checkpoint（下周 W12 要导出 ONNX）

In [ ]:
ckpt_dir = PROJECT_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save({'state_dict': m_pst.state_dict(),
            'config': dict(n_feats=N_FEATS, seq_len=SEQ_LEN,
                           patch_len=16, stride=8,
                           d_model=64, n_heads=4, n_layers=3)},
           ckpt_dir / 'w11_patchtst.pt')
torch.save({'state_dict': m_van.state_dict(),
            'config': dict(n_feats=N_FEATS, seq_len=SEQ_LEN,
                           d_model=64, n_heads=4, n_layers=3)},
           ckpt_dir / 'w11_vanilla.pt')
print('saved:', list(ckpt_dir.glob('w11_*.pt')))

## 10. 反思笔记（自问自答）

1. **PatchTST 为什么有效？**
   - *Patching* 把 `L=256` 的序列降成 `N=31` 个 patch，attention 计算降一个数量级；
   - *Channel-independence* 避免了跨变量的相互干扰——欺诈场景里金额和地域的量纲/分布差异很大，混在一起反而会稀释有效信号。

2. **什么时候该选 PatchTST 而不是 vanilla Transformer？**
   - 序列长（`L >= 128`）、显存吃紧；
   - 变量之间语义/量纲差异大，跨变量 attention 收益有限；
   - 需要快速训练/推理，patching 天然降 token 数。

3. **什么时候 vanilla 反而更好？**
   - 序列短（`L <= 64`）；
   - 变量间耦合强（需要 cross-variate attention）；
   - 目标是重构/生成任务，patching 带来的平均化会损失局部精度。

4. **ProbSparse vs PatchTST 的本质差别？**
   - ProbSparse 是在 **query 数量** 上稀疏（仍在原始 L 维上做 attention）；
   - PatchTST 是在 **token 数量** 上降维（把 L 折成 N=(L-P)/S+1）；
   - 二者可以组合，但组合后实现复杂度急剧上升，官方 PatchTST 没走这条路。

5. **我填写的对比表显示了什么？**
   - *（把 `tbl` 的具体数字摘录到这里，给出你的结论）*